# Assignment 09: Time Series Features from Patient Vitals

`README.md` gives each task's steps and checkpoints; the headings here match its numbering. Run each cell with `Shift+Enter`. When you finish, click **Restart**, then **Run All**, and run `python check_assignment.py` in the terminal.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

print("pandas:", pd.__version__)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

PREPARED_PATH = OUTPUT_DIR / "prepared_vitals.csv"
HOURLY_GRID_PATH = OUTPUT_DIR / "hourly_grid.csv"
TWO_HOUR_PATH = OUTPUT_DIR / "two_hour_summary.csv"
FEATURES_PATH = OUTPUT_DIR / "past_features.csv"
LABS_PATH = OUTPUT_DIR / "lab_availability.csv"
BLOCKS_PATH = OUTPUT_DIR / "chronological_blocks.csv"

print("data folder found:", DATA_DIR.exists())

## Load the data

This cell reads the two supplied files: one row per charted heart rate, and one row per lab order. Run it as it is.

In [ ]:
vitals_raw = pd.read_csv(DATA_DIR / "vitals.csv")
labs_raw = pd.read_csv(DATA_DIR / "labs.csv")
print("vitals:", vitals_raw.shape)
print("labs:", labs_raw.shape)
vitals_raw

## Task 1: Prepare the vitals panel

### 1.1 Parse, localize, and sort the readings

> **Checkpoint: `output/prepared_vitals.csv`**
> Twelve rows under the header `patient_id,recorded_at,heart_rate,source_row`.

In [ ]:
vitals = vitals_raw.copy()
# TODO: parse vitals["recorded_at"] with pd.to_datetime(..., format="%Y-%m-%d %H:%M"),
#   then .dt.tz_localize("America/New_York") and .dt.tz_convert("UTC")
# TODO: sort vitals by ["patient_id", "recorded_at"], then .reset_index(drop=True)
# TODO: add a source_row column of 1s
print("recorded_at dtype:", vitals["recorded_at"].dtype)

# TODO: save vitals to PREPARED_PATH without the row index

vitals

## Task 2: Change the frequency

### 2.1 Build each patient's hourly grid

> **Checkpoint: `output/hourly_grid.csv`**
> Sixteen rows under the header `patient_id,recorded_at,heart_rate,source_row,grid_created,value_missing`.

In [ ]:
print("all on the hour:", vitals["recorded_at"].eq(vitals["recorded_at"].dt.floor("h")).all())

hourly_grid = None  # TODO: vitals.set_index("recorded_at").groupby("patient_id")[["heart_rate", "source_row"]],
#   then .resample("h").asfreq().reset_index()
# TODO: add grid_created: hourly_grid["source_row"].isna()
# TODO: add value_missing: hourly_grid["source_row"].notna() & hourly_grid["heart_rate"].isna()
print("rows:", len(hourly_grid))
print(hourly_grid[["grid_created", "value_missing"]].sum())

# TODO: save hourly_grid to HOURLY_GRID_PATH without the row index

hourly_grid

### 2.2 Summarize two-hour bins

> **Checkpoint: `output/two_hour_summary.csv`**
> Nine rows under the header `patient_id,recorded_at,mean_hr,n_rows`.

In [ ]:
two_hour_summary = None  # TODO: vitals.set_index("recorded_at").groupby("patient_id").resample("2h"),
#   then .agg(mean_hr=("heart_rate", "mean"), n_rows=("source_row", "count")).reset_index()
print("rows:", len(two_hour_summary))
print("readings counted:", two_hour_summary["n_rows"].sum())

# TODO: save two_hour_summary to TWO_HOUR_PATH without the row index

two_hour_summary

## Task 3: Build past-only evidence

### 3.1 Calculate past-only features

> **Checkpoint: `output/past_features.csv`**
> Twelve rows under the header `patient_id,recorded_at,heart_rate,previous_hr,hr_change,mean_prev_2,mean_prev_2h`.

In [ ]:
past_features = vitals[["patient_id", "recorded_at", "heart_rate"]].copy()
by_patient = past_features.groupby("patient_id")["heart_rate"]
# TODO: add previous_hr: by_patient.shift(1)
# TODO: add hr_change: by_patient.diff()
# TODO: add mean_prev_2: by_patient.transform(lambda s: s.shift(1).rolling(2, min_periods=1).mean())

prev_2h = None  # TODO: past_features.set_index("recorded_at").groupby("patient_id")["heart_rate"],
#   then .rolling("2h", closed="left").mean().rename("mean_prev_2h").reset_index()
# TODO: merge prev_2h into past_features on ["patient_id", "recorded_at"] with validate="one_to_one"
print("rows:", len(past_features))

# TODO: save past_features to FEATURES_PATH without the row index

past_features

### 3.2 Check which labs were known at the prediction time

> **Checkpoint: `output/lab_availability.csv`**
> Six rows under the header `patient_id,test,collected_at,resulted_at,available`.

In [ ]:
prediction_time = None  # TODO: pd.Timestamp("2026-01-20 18:00", tz="UTC")

labs = labs_raw.copy()
# TODO: convert labs["collected_at"] and labs["resulted_at"] the way Task 1.1 converts recorded_at
# TODO: add available: labs["resulted_at"] <= prediction_time
print("available:", labs["available"].sum(), "of", len(labs))

# TODO: save labs to LABS_PATH without the row index

labs

### 3.3 Label a chronological holdout

> **Checkpoint: `output/chronological_blocks.csv`**
> Twelve rows under the header `patient_id,recorded_at,heart_rate,source_row,block`.

In [ ]:
chronological_blocks = vitals.copy()
# TODO: add block: np.where(chronological_blocks["recorded_at"] < prediction_time, "earlier", "later_holdout")
print(pd.crosstab(chronological_blocks["patient_id"], chronological_blocks["block"]))

# TODO: save chronological_blocks to BLOCKS_PATH without the row index

chronological_blocks

## Fresh-run check

Click **Restart**, then **Run All**. This cell checks the notebook's own results; `python check_assignment.py` checks the six saved files.

In [ ]:
assert str(vitals["recorded_at"].dt.tz) == "UTC", "Task 1.1: localize to America/New_York, then convert to UTC"
assert vitals["source_row"].eq(1).all(), "Task 1.1: add a source_row column of 1s"
assert len(hourly_grid) == 16, "Task 2.1: one row per hour from each patient's first reading to their last"
assert hourly_grid["grid_created"].sum() == 4, "Task 2.1: grid_created is True on the 4 hours with no charted row"
assert hourly_grid["value_missing"].sum() == 1, "Task 2.1: value_missing is True on the 1 charted row with no heart rate"
assert len(two_hour_summary) == 9, "Task 2.2: one row per patient and two-hour bin"
assert two_hour_summary["n_rows"].sum() == 12, "Task 2.2: n_rows counts source_row, so all 12 readings count"
assert len(past_features) == len(vitals), "Task 3.1: the merge keeps one row per reading"
assert labs["available"].sum() == 3, "Task 3.2: available is resulted_at <= prediction_time"
assert chronological_blocks["block"].eq("earlier").sum() == 8, "Task 3.3: rows before prediction_time are earlier"
for path in [PREPARED_PATH, HOURLY_GRID_PATH, TWO_HOUR_PATH, FEATURES_PATH, LABS_PATH, BLOCKS_PATH]:
    assert path.exists(), f"{path} is missing; run the cell that saves it"

print("Fresh-run check passed")